In [ ]:
import pandas as pd
import os
import json

# ==========================
# 1. Load alerts file
# ==========================
alerts = pd.read_csv("alerts_data.csv")

print("Columns in alerts_data.csv:")
print(alerts.columns)
print()

# ==========================
# 2. Filter ground-truth subset
#    (regression alerts only)
# ==========================
df = alerts[alerts["single_alert_is_regression"] == True].copy()

# OPTIONAL: remove invalid/noise statuses
# Example: df = df[df["single_alert_status"] != 0]

print(f"Total regression alerts: {len(df)}")

# ==========================
# 3. Detect timestamp column
# ==========================
timestamp_candidates = [
    "push_timestamp",
    "alert_summary_creation_timestamp",
    "timestamp",
    "alert_timestamp"
]

timestamp_col = None
for col in timestamp_candidates:
    if col in df.columns:
        timestamp_col = col
        break

if timestamp_col is None:
    raise ValueError("No timestamp column found.")

print("Using timestamp column:", timestamp_col)

df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
df = df.dropna(subset=["signature_id", timestamp_col])

# ==========================
# 4. Group by signature
# ==========================
grouped = (
    df.groupby("signature_id")[timestamp_col]
    .apply(lambda s: sorted(set(s.dt.strftime("%Y-%m-%d %H:%M:%S"))))
)

rows = []

for signature, timestamps in grouped.items():
    ts_filename = f"{signature}_timeseries_data.csv"
    ts_exists = os.path.exists(ts_filename)

    rows.append({
        "signature_id": signature,
        "timeseries_filename": ts_filename if ts_exists else "",
        "alert_timestamps": json.dumps(timestamps),
        "num_alerts": len(timestamps),
        "timeseries_found": ts_exists
    })

eval_df = pd.DataFrame(rows).sort_values(
    ["timeseries_found", "num_alerts"],
    ascending=[False, False]
)

# ==========================
# 5. Save output
# ==========================
eval_df.to_csv("mozilla_eval_signatures.csv", index=False)

print()
print("Done.")
print(f"Total signatures: {len(eval_df)}")
print(f"Timeseries files found: {eval_df['timeseries_found'].sum()}")

Columns in alerts_data.csv:
Index(['single_alert_new_value', 'single_alert_classifier',
       'alert_summary_prev_push_revision', 'alert_summary_id',
       'single_alert_amount_pct', 'alert_summary_performance_tags',
       'single_alert_prev_value', 'alert_summary_first_triaged',
       'single_alert_series_signature_has_subtests', 'alert_summary_framework',
       'single_alert_series_signature_framework_id',
       'single_alert_series_signature_suite_public_name',
       'alert_summary_related_alerts',
       'single_alert_taskcluster_metadata_retry_id',
       'alert_summary_prev_push_id', 'alert_summary_status',
       'alert_summary_push_id', 'single_alert_status',
       'alert_summary_assignee_username',
       'single_alert_series_signature_signature_hash',
       'single_alert_prev_taskcluster_metadata_task_id',
       'single_alert_series_signature_machine_platform',
       'alert_summary_bug_number', 'alert_summary_repository',
       'alert_summary_bug_due_date',
      

In [ ]:
# ==========================
# Step 3 — Convert each signature CSV to a clean vector y[0..T]
# Colab-ready: assumes all *_timeseries_data.csv are in the current directory
# Outputs:
#   vectors/<signature>.npy
#   vectors/<signature>_meta.json
#   vectors/index.json
# ==========================

import os
import re
import glob
import json
import numpy as np
import pandas as pd

# ---------- JSON sanitizer (fixes Timestamp not JSON serializable) ----------
def json_sanitize(obj):
    """Convert pandas/numpy types to JSON-serializable Python types."""
    if obj is None:
        return None

    # pandas timestamp
    if isinstance(obj, pd.Timestamp):
        return None if pd.isna(obj) else obj.isoformat()

    # pandas NA/NaT handling
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass

    # numpy scalars
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    # numpy arrays
    if isinstance(obj, np.ndarray):
        return obj.tolist()

    # containers
    if isinstance(obj, dict):
        return {k: json_sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_sanitize(v) for v in obj]

    return obj


def pick_timestamp_column(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns and df[c].notna().any():
            return c
    return None


def signature_csv_to_vector(
    csv_path: str,
    value_col: str = "value",
    ts_col: str = "push_timestamp",
    push_id_col: str = "push_id",
    rev_col: str = "revision",
    agg: str = "mean",           # "mean" or "median"
    dedupe_on: str = "revision"  # "revision" or "push_id" or None
):
    df = pd.read_csv(csv_path)

    if value_col not in df.columns:
        raise ValueError(f"Missing numeric value column '{value_col}' in {csv_path}.")

    # --- pick sort key (prefer timestamp, else push_id, else revision) ---
    # Note: even if push_timestamp exists, it might be all-null for some files
    sort_key = None
    if ts_col in df.columns and df[ts_col].notna().any():
        sort_key = ts_col
        df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    elif push_id_col in df.columns and df[push_id_col].notna().any():
        sort_key = push_id_col
    elif rev_col in df.columns and df[rev_col].notna().any():
        sort_key = rev_col
    else:
        raise ValueError(
            f"No usable time key found in {csv_path} (tried {ts_col}, {push_id_col}, {rev_col})."
        )

    # keep essential cols if present
    keep = [c for c in [value_col, ts_col, push_id_col, rev_col] if c in df.columns]
    df = df[keep].copy()

    # numeric value
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[value_col]).copy()

    # --- optional aggregation (if multiple rows per revision/push_id) ---
    group_key = None
    if dedupe_on == "revision" and rev_col in df.columns and df[rev_col].notna().any():
        group_key = rev_col
    elif dedupe_on == "push_id" and push_id_col in df.columns and df[push_id_col].notna().any():
        group_key = push_id_col
    else:
        group_key = None

    if group_key:
        # ordering key per group (min time / id)
        if sort_key == ts_col and ts_col in df.columns:
            order_series = df.groupby(group_key)[ts_col].min()
        else:
            order_series = df.groupby(group_key)[sort_key].min()

        # aggregate
        if agg == "median":
            agg_vals = df.groupby(group_key)[value_col].median()
        else:
            agg_vals = df.groupby(group_key)[value_col].mean()

        out = pd.DataFrame({
            group_key: agg_vals.index,
            value_col: agg_vals.values,
            "_order_key": order_series.loc[agg_vals.index].values
        })

        # attach representative metadata
        if ts_col in df.columns:
            out[ts_col] = df.groupby(group_key)[ts_col].min().loc[agg_vals.index].values
        if push_id_col in df.columns:
            out[push_id_col] = df.groupby(group_key)[push_id_col].min().loc[agg_vals.index].values
        if rev_col in df.columns:
            out[rev_col] = df.groupby(group_key)[rev_col].min().loc[agg_vals.index].values

        out = out.sort_values("_order_key").drop(columns=["_order_key"]).reset_index(drop=True)

    else:
        # no grouping: just sort
        out = df.sort_values(sort_key).reset_index(drop=True)

    # final vector
    y = out[value_col].to_numpy(dtype=float)

    # meta rows (may include Timestamps -> sanitize later)
    meta = out.to_dict(orient="records")

    return y, meta, sort_key, group_key


# ==========================
# Batch convert ALL signature files in the folder
# ==========================
os.makedirs("vectors", exist_ok=True)

index = []
files = sorted(glob.glob("*_timeseries_data.csv"))

print(f"Found {len(files)} timeseries files.")

for csv_path in files:
    base = os.path.basename(csv_path)

    # signature_id parsing: handles "4754836_timeseries_data.csv"
    m = re.match(r"^(.+)_timeseries_data\.csv$", base)
    signature_id = m.group(1) if m else base.replace("_timeseries_data.csv", "")

    y, meta, sort_key, group_key = signature_csv_to_vector(
        csv_path,
        agg="mean",            # change to "median" if desired
        dedupe_on="revision"   # change to "push_id" or None if needed
    )

    np.save(f"vectors/{signature_id}.npy", y)

    with open(f"vectors/{signature_id}_meta.json", "w") as f:
        json.dump(
            json_sanitize({
                "signature_id": signature_id,
                "sort_key": sort_key,
                "group_key": group_key,
                "rows": meta
            }),
            f,
            ensure_ascii=False,
            indent=2
        )

    index.append({
        "signature_id": signature_id,
        "T": int(len(y)),
        "sort_key": sort_key,
        "group_key": group_key,
        "source_csv": base
    })

with open("vectors/index.json", "w") as f:
    json.dump(index, f, indent=2)

print("Done.")
print(f"Converted {len(index)} files into vectors/")
print("Example index entry:", index[0] if index else None)

Found 10 timeseries files.
Done.
Converted 10 files into vectors/
Example index entry: {'signature_id': '4653372', 'T': 571, 'sort_key': 'push_timestamp', 'group_key': 'revision', 'source_csv': '4653372_timeseries_data.csv'}


In [ ]:
import shutil
from google.colab import files

# Create zip archive
shutil.make_archive("vectors", "zip", "vectors")

# Download it
files.download("vectors.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================
# Step 4 — ARIMA + anomalyScore (Algorithm 3 controller) — COLAB CELL (FIXED)
# Fix: normalize signature IDs so "4754836.0" matches "4754836"
# Also: process all vectors even if not present in eval (still runs ARIMA)
# ==========================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# ---------- SETTINGS ----------
EVAL_CSV = "mozilla_eval_signatures.csv"
VECTORS_DIR = "vectors"
OUT_CSV = "mozilla_results.csv"
PLOTS_DIR = "plots"
PLOT_N = 10

# Algorithm 3 parameters
BETA = 0.30
THRESHOLD_ANOMALY = 0.30

# Dynamic Threshold_arima heuristic
K_SIGMA = 3.0
THR_WINDOW = 30
MIN_THRESHOLD_ARIMA = 1e-6

# ARIMA settings
ARIMA_ORDER = (1, 1, 1)
MIN_HISTORY = 20
FALLBACK_TO_LAST_VALUE = True

os.makedirs(PLOTS_DIR, exist_ok=True)

# ---------- Helpers ----------
def normalize_sig(x) -> str:
    """
    Make signature IDs comparable between:
      - CSV values (may be numeric => '4754836.0')
      - filenames (usually '4754836')
    """
    if x is None:
        return ""
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return ""
    # If it's numeric-like, coerce through float->int safely
    try:
        f = float(s)
        if math.isfinite(f):
            i = int(f)
            # only accept if it doesn't lose info (e.g., 123.0)
            if abs(f - i) < 1e-9:
                return str(i)
    except Exception:
        pass
    return s

def safe_float(x):
    try:
        if x is None:
            return None
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return None
        return v
    except Exception:
        return None

def update_anomaly_score(prev_score: float, is_anomaly: int, beta: float) -> float:
    return (beta * float(is_anomaly)) + ((1.0 - beta) * float(prev_score))

def fit_predict_arima(history: np.ndarray, order=(1,1,1)) -> float | None:
    try:
        model = ARIMA(history, order=order, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit()
        fc = res.forecast(steps=1)
        return safe_float(fc[0])
    except Exception:
        return None

def threshold_arima_dynamic(delta_hist, k_sigma, window, min_thr):
    if len(delta_hist) == 0:
        return max(min_thr, 0.0)
    w = delta_hist[-window:] if len(delta_hist) > window else delta_hist
    s = float(np.std(w)) if len(w) >= 2 else float(np.std(delta_hist))
    return max(min_thr, k_sigma * s)

def plot_signature(df_sig: pd.DataFrame, out_path: str, title: str):
    t = df_sig["t"].to_numpy()
    actual = df_sig["actual"].to_numpy(dtype=float)
    predicted = df_sig["predicted"].to_numpy(dtype=float)
    decision = df_sig["decision"].to_numpy(dtype=float)
    anomaly_score = df_sig["anomalyScore"].to_numpy(dtype=float)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(t, actual, label="Actual")
    axes[0].plot(t, predicted, label="Predicted")
    axes[0].set_ylabel("Value")
    axes[0].set_title(title)
    axes[0].legend(loc="best")

    axes[1].step(t, decision, where="post", label="Decision (Enable=1)")
    axes[1].plot(t, anomaly_score, label="anomalyScore")
    axes[1].set_ylabel("Decision / anomalyScore")
    axes[1].set_xlabel("t")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].legend(loc="best")
    axes[1].set_title("Adaptive Decision")

    plt.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

# ---------- Load eval ----------
eval_df = pd.read_csv(EVAL_CSV)
if "signature_id" not in eval_df.columns:
    raise ValueError(f"{EVAL_CSV} must include 'signature_id' column.")

eval_df["sig_norm"] = eval_df["signature_id"].apply(normalize_sig)

# quick debug: show if 4754836 exists in eval after normalization
print("Does eval contain 4754836?")
print((eval_df["sig_norm"] == "4754836").any())
if (eval_df["sig_norm"] == "4754836").any():
    print("Rows in eval for 4754836:", len(eval_df[eval_df["sig_norm"] == "4754836"]))

# ---------- List available vectors ----------
if not os.path.isdir(VECTORS_DIR):
    raise FileNotFoundError(f"Missing '{VECTORS_DIR}/' folder. Run Step 3 first.")

available = sorted([f[:-4] for f in os.listdir(VECTORS_DIR) if f.endswith(".npy")])
available_norm = [normalize_sig(x) for x in available]
available_set = set(available_norm)

print("\nTotal signatures in eval file:", len(eval_df))
print("Vectors available (.npy):", len(available))
print("Available vector IDs:", available_norm)

# Build a lookup from normalized id -> eval row (first match)
eval_lookup = {}
for _, r in eval_df.iterrows():
    sid = r["sig_norm"]
    if sid and sid not in eval_lookup:
        eval_lookup[sid] = r  # keep first occurrence

# We will process ALL available vectors (2 in your case)
to_process = available_norm
print("\nSignatures to process (from vectors):", to_process)

# ---------- Run ARIMA controller ----------
all_rows = []
plotted = 0

for sig_norm in to_process:
    npy_path = os.path.join(VECTORS_DIR, f"{sig_norm}.npy")
    if not os.path.exists(npy_path):
        # fallback: maybe original filename wasn't normalized; try finding it
        # (rare, but safe)
        candidates = [f for f in os.listdir(VECTORS_DIR) if f.endswith(".npy") and normalize_sig(f[:-4]) == sig_norm]
        if not candidates:
            print(f"[WARN] Missing npy for {sig_norm}")
            continue
        npy_path = os.path.join(VECTORS_DIR, candidates[0])

    y = np.load(npy_path).astype(float)
    T = len(y)

    anomaly_score = 0.0
    delta_hist = []
    rows = []

    for t in range(T):
        actual = safe_float(y[t])

        if t < MIN_HISTORY or actual is None:
            rows.append({
                "signature_id": sig_norm,
                "t": t,
                "actual": actual,
                "predicted": np.nan,
                "delta": np.nan,
                "threshold_arima": np.nan,
                "isAnomaly": 0,
                "anomalyScore": anomaly_score,
                "decision": 0,
            })
            continue

        history = y[:t]
        pred = fit_predict_arima(history, order=ARIMA_ORDER)
        if pred is None and FALLBACK_TO_LAST_VALUE:
            pred = safe_float(history[-1])

        if pred is None or actual is None:
            rows.append({
                "signature_id": sig_norm,
                "t": t,
                "actual": actual,
                "predicted": np.nan if pred is None else pred,
                "delta": np.nan,
                "threshold_arima": np.nan,
                "isAnomaly": 0,
                "anomalyScore": anomaly_score,
                "decision": 0,
            })
            continue

        delta = abs(actual - pred)
        thr_arima = threshold_arima_dynamic(delta_hist, K_SIGMA, THR_WINDOW, MIN_THRESHOLD_ARIMA)

        is_anomaly = 1 if delta > thr_arima else 0
        anomaly_score = update_anomaly_score(anomaly_score, is_anomaly, BETA)
        decision = 1 if anomaly_score >= THRESHOLD_ANOMALY else 0

        delta_hist.append(delta)

        rows.append({
            "signature_id": sig_norm,
            "t": t,
            "actual": actual,
            "predicted": pred,
            "delta": delta,
            "threshold_arima": thr_arima,
            "isAnomaly": is_anomaly,
            "anomalyScore": anomaly_score,
            "decision": decision,
        })

    df_sig = pd.DataFrame(rows)

    # Attach eval metadata if present (alert timestamps, etc.)
    if sig_norm in eval_lookup:
        r = eval_lookup[sig_norm]
        for col in eval_df.columns:
            if col not in ("signature_id", "sig_norm"):
                df_sig[col] = r[col]
        df_sig["in_eval_file"] = True
    else:
        df_sig["in_eval_file"] = False

    all_rows.append(df_sig)

    # Plot
    if plotted < PLOT_N:
        out_plot = os.path.join(PLOTS_DIR, f"{sig_norm}_prediction_and_decision.png")
        plot_signature(df_sig, out_plot, title=f"Signature {sig_norm} — ARIMA Prediction")
        plotted += 1

# ---------- Save ----------
out_df = pd.concat(all_rows, ignore_index=True)
out_df.to_csv(OUT_CSV, index=False)

print("\n==== DONE ====")
print(f"Processed signatures: {out_df['signature_id'].nunique()}")
print(f"Rows written: {len(out_df)}")
print(f"Saved: {OUT_CSV}")
print(f"Plots saved to: {PLOTS_DIR}/ (plotted {plotted})")

valid = out_df.dropna(subset=["delta", "threshold_arima"]).copy()
if len(valid) > 0:
    print("\nSanity checks (valid rows only):")
    print("isAnomaly rate:", float(valid["isAnomaly"].mean()))
    print("decision rate:", float(valid["decision"].mean()))
    print("\ndelta summary:\n", valid["delta"].describe())
    print("\nthreshold_arima summary:\n", valid["threshold_arima"].describe())

Does eval contain 4754836?
False

Total signatures in eval file: 3764
Vectors available (.npy): 10
Available vector IDs: ['4653372', '4653777', '4653804', '4653842', '4653876', '4653891', '4653997', '4654029', '4677930', '4754836']

Signatures to process (from vectors): ['4653372', '4653777', '4653804', '4653842', '4653876', '4653891', '4653997', '4654029', '4677930', '4754836']


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



==== DONE ====
Processed signatures: 10
Rows written: 5332
Saved: mozilla_results.csv
Plots saved to: plots/ (plotted 10)

Sanity checks (valid rows only):
isAnomaly rate: 0.09119251753702261
decision rate: 0.11847233047544817

delta summary:
 count    5132.000000
mean       19.565177
std        46.470520
min         0.000282
25%         2.467882
50%         7.977943
75%        21.358573
max      1632.609386
Name: delta, dtype: float64

threshold_arima summary:
 count    5132.000000
mean       55.284064
std       107.868331
min         0.000001
25%        12.185622
50%        28.473992
75%        60.040095
max      1042.715355
Name: threshold_arima, dtype: float64


In [ ]:
# ==========================
# Step 4 — LAST (Naïve) + anomalyScore (Algorithm 3 controller) — COLAB CELL
# Replaces ARIMA with LAST: y_hat[t] = y[t-1]
# Keeps the same delta / dynamic threshold / anomalyScore / decision logic
# ==========================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- SETTINGS ----------
EVAL_CSV = "mozilla_eval_signatures.csv"
VECTORS_DIR = "vectors"
OUT_CSV = "mozilla_results_LAST.csv"
PLOTS_DIR = "plots_LAST"
PLOT_N = 10

# Algorithm 3 parameters
BETA = 0.30
THRESHOLD_ANOMALY = 0.30

# Dynamic threshold heuristic (same as before)
K_SIGMA = 3.0
THR_WINDOW = 30
MIN_THRESHOLD_BASELINE = 1e-6

# Baseline settings
MIN_HISTORY = 2  # need at least y[t-1]

os.makedirs(PLOTS_DIR, exist_ok=True)

# ---------- Helpers ----------
def normalize_sig(x) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return ""
    try:
        f = float(s)
        if math.isfinite(f):
            i = int(f)
            if abs(f - i) < 1e-9:
                return str(i)
    except Exception:
        pass
    return s

def safe_float(x):
    try:
        if x is None:
            return None
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return None
        return v
    except Exception:
        return None

def update_anomaly_score(prev_score: float, is_anomaly: int, beta: float) -> float:
    return (beta * float(is_anomaly)) + ((1.0 - beta) * float(prev_score))

def threshold_dynamic(delta_hist, k_sigma, window, min_thr):
    if len(delta_hist) == 0:
        return max(min_thr, 0.0)
    w = delta_hist[-window:] if len(delta_hist) > window else delta_hist
    s = float(np.std(w)) if len(w) >= 2 else float(np.std(delta_hist))
    return max(min_thr, k_sigma * s)

def plot_signature(df_sig: pd.DataFrame, out_path: str, title: str):
    t = df_sig["t"].to_numpy()
    actual = df_sig["actual"].to_numpy(dtype=float)
    predicted = df_sig["predicted"].to_numpy(dtype=float)
    decision = df_sig["decision"].to_numpy(dtype=float)
    anomaly_score = df_sig["anomalyScore"].to_numpy(dtype=float)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(t, actual, label="Actual")
    axes[0].plot(t, predicted, label="Predicted (LAST)")
    axes[0].set_ylabel("Value")
    axes[0].set_title(title)
    axes[0].legend(loc="best")

    axes[1].step(t, decision, where="post", label="Decision (Enable=1)")
    axes[1].plot(t, anomaly_score, label="anomalyScore")
    axes[1].set_ylabel("Decision / anomalyScore")
    axes[1].set_xlabel("t")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].legend(loc="best")
    axes[1].set_title("Adaptive Decision")

    plt.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

# ---------- Load eval ----------
eval_df = pd.read_csv(EVAL_CSV)
if "signature_id" not in eval_df.columns:
    raise ValueError(f"{EVAL_CSV} must include 'signature_id' column.")

eval_df["sig_norm"] = eval_df["signature_id"].apply(normalize_sig)

print("Does eval contain 4754836?")
print((eval_df["sig_norm"] == "4754836").any())
if (eval_df["sig_norm"] == "4754836").any():
    print("Rows in eval for 4754836:", len(eval_df[eval_df["sig_norm"] == "4754836"]))

# ---------- List available vectors ----------
if not os.path.isdir(VECTORS_DIR):
    raise FileNotFoundError(f"Missing '{VECTORS_DIR}/' folder. Run Step 3 first.")

available = sorted([f[:-4] for f in os.listdir(VECTORS_DIR) if f.endswith(".npy")])
available_norm = [normalize_sig(x) for x in available]
print("\nVectors available (.npy):", len(available))
print("Available vector IDs:", available_norm)

# Build a lookup from normalized id -> eval row (first match)
eval_lookup = {}
for _, r in eval_df.iterrows():
    sid = r["sig_norm"]
    if sid and sid not in eval_lookup:
        eval_lookup[sid] = r

to_process = available_norm
print("\nSignatures to process (from vectors):", to_process)

# ---------- Run LAST controller ----------
all_rows = []
plotted = 0

for sig_norm in to_process:
    npy_path = os.path.join(VECTORS_DIR, f"{sig_norm}.npy")
    if not os.path.exists(npy_path):
        candidates = [f for f in os.listdir(VECTORS_DIR) if f.endswith(".npy") and normalize_sig(f[:-4]) == sig_norm]
        if not candidates:
            print(f"[WARN] Missing npy for {sig_norm}")
            continue
        npy_path = os.path.join(VECTORS_DIR, candidates[0])

    y = np.load(npy_path).astype(float)
    T = len(y)

    anomaly_score = 0.0
    delta_hist = []
    rows = []

    for t in range(T):
        actual = safe_float(y[t])

        if t < MIN_HISTORY or actual is None:
            rows.append({
                "signature_id": sig_norm,
                "t": t,
                "actual": actual,
                "predicted": np.nan,
                "delta": np.nan,
                "threshold_baseline": np.nan,
                "isAnomaly": 0,
                "anomalyScore": anomaly_score,
                "decision": 0,
            })
            continue

        # LAST baseline prediction
        pred = safe_float(y[t - 1])

        if pred is None or actual is None:
            rows.append({
                "signature_id": sig_norm,
                "t": t,
                "actual": actual,
                "predicted": np.nan if pred is None else pred,
                "delta": np.nan,
                "threshold_baseline": np.nan,
                "isAnomaly": 0,
                "anomalyScore": anomaly_score,
                "decision": 0,
            })
            continue

        delta = abs(actual - pred)
        thr = threshold_dynamic(delta_hist, K_SIGMA, THR_WINDOW, MIN_THRESHOLD_BASELINE)

        is_anomaly = 1 if delta > thr else 0
        anomaly_score = update_anomaly_score(anomaly_score, is_anomaly, BETA)
        decision = 1 if anomaly_score >= THRESHOLD_ANOMALY else 0

        delta_hist.append(delta)

        rows.append({
            "signature_id": sig_norm,
            "t": t,
            "actual": actual,
            "predicted": pred,
            "delta": delta,
            "threshold_baseline": thr,
            "isAnomaly": is_anomaly,
            "anomalyScore": anomaly_score,
            "decision": decision,
        })

    df_sig = pd.DataFrame(rows)

    # Attach eval metadata if present
    if sig_norm in eval_lookup:
        r = eval_lookup[sig_norm]
        for col in eval_df.columns:
            if col not in ("signature_id", "sig_norm"):
                df_sig[col] = r[col]
        df_sig["in_eval_file"] = True
    else:
        df_sig["in_eval_file"] = False

    all_rows.append(df_sig)

    # Plot
    if plotted < PLOT_N:
        out_plot = os.path.join(PLOTS_DIR, f"{sig_norm}_prediction_and_decision.png")
        plot_signature(df_sig, out_plot, title=f"Signature {sig_norm} — LAST Prediction")
        plotted += 1

# ---------- Save ----------
out_df = pd.concat(all_rows, ignore_index=True)
out_df.to_csv(OUT_CSV, index=False)

print("\n==== DONE (LAST) ====")
print(f"Processed signatures: {out_df['signature_id'].nunique()}")
print(f"Rows written: {len(out_df)}")
print(f"Saved: {OUT_CSV}")
print(f"Plots saved to: {PLOTS_DIR}/ (plotted {plotted})")

valid = out_df.dropna(subset=["delta", "threshold_baseline"]).copy()
if len(valid) > 0:
    print("\nSanity checks (valid rows only):")
    print("isAnomaly rate:", float(valid["isAnomaly"].mean()))
    print("decision rate:", float(valid["decision"].mean()))
    print("\ndelta summary:\n", valid["delta"].describe())
    print("\nthreshold_baseline summary:\n", valid["threshold_baseline"].describe())

Does eval contain 4754836?
False

Vectors available (.npy): 10
Available vector IDs: ['4653372', '4653777', '4653804', '4653842', '4653876', '4653891', '4653997', '4654029', '4677930', '4754836']

Signatures to process (from vectors): ['4653372', '4653777', '4653804', '4653842', '4653876', '4653891', '4653997', '4654029', '4677930', '4754836']

==== DONE (LAST) ====
Processed signatures: 10
Rows written: 5332
Saved: mozilla_results_LAST.csv
Plots saved to: plots_LAST/ (plotted 10)

Sanity checks (valid rows only):
isAnomaly rate: 0.10353915662650602
decision rate: 0.14250753012048192

delta summary:
 count    5312.000000
mean       23.943876
std        57.246170
min         0.000000
25%         3.000000
50%         9.580000
75%        25.100000
max      1627.300000
Name: delta, dtype: float64

threshold_baseline summary:
 count    5312.000000
mean       67.379290
std       133.712681
min         0.000001
25%        13.508760
50%        33.345285
75%        74.470707
max      1212.91980

In [ ]:
# ==========================
# Step 4 — SMA (Rolling Mean) + anomalyScore (Algorithm 3 controller) — COLAB CELL
# Replaces ARIMA with SMA: y_hat[t] = mean(y[t-w : t])
# Keeps the same delta / dynamic threshold / anomalyScore / decision logic
# ==========================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- SETTINGS ----------
EVAL_CSV = "mozilla_eval_signatures.csv"
VECTORS_DIR = "vectors"
OUT_CSV = "mozilla_results_SMA.csv"
PLOTS_DIR = "plots_SMA"
PLOT_N = 10

# Algorithm 3 parameters
BETA = 0.30
THRESHOLD_ANOMALY = 0.30

# Dynamic threshold heuristic (same as before)
K_SIGMA = 3.0
THR_WINDOW = 30
MIN_THRESHOLD_BASELINE = 1e-6

# SMA baseline settings
SMA_WINDOW = 20
MIN_HISTORY = SMA_WINDOW  # require at least w points before predicting

os.makedirs(PLOTS_DIR, exist_ok=True)

# ---------- Helpers ----------
def normalize_sig(x) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return ""
    try:
        f = float(s)
        if math.isfinite(f):
            i = int(f)
            if abs(f - i) < 1e-9:
                return str(i)
    except Exception:
        pass
    return s

def safe_float(x):
    try:
        if x is None:
            return None
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return None
        return v
    except Exception:
        return None

def update_anomaly_score(prev_score: float, is_anomaly: int, beta: float) -> float:
    return (beta * float(is_anomaly)) + ((1.0 - beta) * float(prev_score))

def threshold_dynamic(delta_hist, k_sigma, window, min_thr):
    if len(delta_hist) == 0:
        return max(min_thr, 0.0)
    w = delta_hist[-window:] if len(delta_hist) > window else delta_hist
    s = float(np.std(w)) if len(w) >= 2 else float(np.std(delta_hist))
    return max(min_thr, k_sigma * s)

def plot_signature(df_sig: pd.DataFrame, out_path: str, title: str):
    t = df_sig["t"].to_numpy()
    actual = df_sig["actual"].to_numpy(dtype=float)
    predicted = df_sig["predicted"].to_numpy(dtype=float)
    decision = df_sig["decision"].to_numpy(dtype=float)
    anomaly_score = df_sig["anomalyScore"].to_numpy(dtype=float)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(t, actual, label="Actual")
    axes[0].plot(t, predicted, label=f"Predicted (SMA w={SMA_WINDOW})")
    axes[0].set_ylabel("Value")
    axes[0].set_title(title)
    axes[0].legend(loc="best")

    axes[1].step(t, decision, where="post", label="Decision (Enable=1)")
    axes[1].plot(t, anomaly_score, label="anomalyScore")
    axes[1].set_ylabel("Decision / anomalyScore")
    axes[1].set_xlabel("t")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].legend(loc="best")
    axes[1].set_title("Adaptive Decision")

    plt.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

# ---------- Load eval ----------
eval_df = pd.read_csv(EVAL_CSV)
if "signature_id" not in eval_df.columns:
    raise ValueError(f"{EVAL_CSV} must include 'signature_id' column.")

eval_df["sig_norm"] = eval_df["signature_id"].apply(normalize_sig)

print("Does eval contain 4754836?")
print((eval_df["sig_norm"] == "4754836").any())
if (eval_df["sig_norm"] == "4754836").any():
    print("Rows in eval for 4754836:", len(eval_df[eval_df["sig_norm"] == "4754836"]))

# ---------- List available vectors ----------
if not os.path.isdir(VECTORS_DIR):
    raise FileNotFoundError(f"Missing '{VECTORS_DIR}/' folder. Run Step 3 first.")

available = sorted([f[:-4] for f in os.listdir(VECTORS_DIR) if f.endswith(".npy")])
available_norm = [normalize_sig(x) for x in available]
print("\nVectors available (.npy):", len(available))
print("Available vector IDs:", available_norm)

# Build a lookup from normalized id -> eval row (first match)
eval_lookup = {}
for _, r in eval_df.iterrows():
    sid = r["sig_norm"]
    if sid and sid not in eval_lookup:
        eval_lookup[sid] = r

to_process = available_norm
print("\nSignatures to process (from vectors):", to_process)

# ---------- Run SMA controller ----------
all_rows = []
plotted = 0

for sig_norm in to_process:
    npy_path = os.path.join(VECTORS_DIR, f"{sig_norm}.npy")
    if not os.path.exists(npy_path):
        candidates = [f for f in os.listdir(VECTORS_DIR) if f.endswith(".npy") and normalize_sig(f[:-4]) == sig_norm]
        if not candidates:
            print(f"[WARN] Missing npy for {sig_norm}")
            continue
        npy_path = os.path.join(VECTORS_DIR, candidates[0])

    y = np.load(npy_path).astype(float)
    T = len(y)

    anomaly_score = 0.0
    delta_hist = []
    rows = []

    for t in range(T):
        actual = safe_float(y[t])

        if t < MIN_HISTORY or actual is None:
            rows.append({
                "signature_id": sig_norm,
                "t": t,
                "actual": actual,
                "predicted": np.nan,
                "delta": np.nan,
                "threshold_baseline": np.nan,
                "isAnomaly": 0,
                "anomalyScore": anomaly_score,
                "decision": 0,
            })
            continue

        # SMA prediction: mean of the last window values up to t-1
        window = y[t - SMA_WINDOW : t]
        pred = safe_float(np.mean(window))

        if pred is None or actual is None:
            rows.append({
                "signature_id": sig_norm,
                "t": t,
                "actual": actual,
                "predicted": np.nan if pred is None else pred,
                "delta": np.nan,
                "threshold_baseline": np.nan,
                "isAnomaly": 0,
                "anomalyScore": anomaly_score,
                "decision": 0,
            })
            continue

        delta = abs(actual - pred)
        thr = threshold_dynamic(delta_hist, K_SIGMA, THR_WINDOW, MIN_THRESHOLD_BASELINE)

        is_anomaly = 1 if delta > thr else 0
        anomaly_score = update_anomaly_score(anomaly_score, is_anomaly, BETA)
        decision = 1 if anomaly_score >= THRESHOLD_ANOMALY else 0

        delta_hist.append(delta)

        rows.append({
            "signature_id": sig_norm,
            "t": t,
            "actual": actual,
            "predicted": pred,
            "delta": delta,
            "threshold_baseline": thr,
            "isAnomaly": is_anomaly,
            "anomalyScore": anomaly_score,
            "decision": decision,
        })

    df_sig = pd.DataFrame(rows)

    # Attach eval metadata if present
    if sig_norm in eval_lookup:
        r = eval_lookup[sig_norm]
        for col in eval_df.columns:
            if col not in ("signature_id", "sig_norm"):
                df_sig[col] = r[col]
        df_sig["in_eval_file"] = True
    else:
        df_sig["in_eval_file"] = False

    all_rows.append(df_sig)

    # Plot
    if plotted < PLOT_N:
        out_plot = os.path.join(PLOTS_DIR, f"{sig_norm}_prediction_and_decision.png")
        plot_signature(df_sig, out_plot, title=f"Signature {sig_norm} — SMA Prediction")
        plotted += 1

# ---------- Save ----------
out_df = pd.concat(all_rows, ignore_index=True)
out_df.to_csv(OUT_CSV, index=False)

print("\n==== DONE (SMA) ====")
print(f"Processed signatures: {out_df['signature_id'].nunique()}")
print(f"Rows written: {len(out_df)}")
print(f"Saved: {OUT_CSV}")
print(f"Plots saved to: {PLOTS_DIR}/ (plotted {plotted})")

valid = out_df.dropna(subset=["delta", "threshold_baseline"]).copy()
if len(valid) > 0:
    print("\nSanity checks (valid rows only):")
    print("isAnomaly rate:", float(valid["isAnomaly"].mean()))
    print("decision rate:", float(valid["decision"].mean()))
    print("\ndelta summary:\n", valid["delta"].describe())
    print("\nthreshold_baseline summary:\n", valid["threshold_baseline"].describe())

Does eval contain 4754836?
False

Vectors available (.npy): 10
Available vector IDs: ['4653372', '4653777', '4653804', '4653842', '4653876', '4653891', '4653997', '4654029', '4677930', '4754836']

Signatures to process (from vectors): ['4653372', '4653777', '4653804', '4653842', '4653876', '4653891', '4653997', '4654029', '4677930', '4754836']

==== DONE (SMA) ====
Processed signatures: 10
Rows written: 5332
Saved: mozilla_results_SMA.csv
Plots saved to: plots_SMA/ (plotted 10)

Sanity checks (valid rows only):
isAnomaly rate: 0.09820732657833203
decision rate: 0.13074824629773968

delta summary:
 count    5132.000000
mean       21.130962
std        45.469387
min         0.000500
25%         2.415000
50%         8.695000
75%        23.726250
max      1634.687500
Name: delta, dtype: float64

threshold_baseline summary:
 count    5132.000000
mean       57.397812
std       100.574740
min         0.000001
25%        11.875548
50%        29.450386
75%        66.089656
max       877.710690
N

In [ ]:
import pandas as pd
import numpy as np
import os, json

RESULTS_CSV = "mozilla_results.csv"
ALERTS_CSV  = "alerts_data.csv"
META_DIR    = "vectors"
TOL = 5

def norm_sig(x):
    try: return str(int(float(x)))
    except: return str(x).strip()

def extract_intervals(ts):
    """ts = sorted list of t where decision==1 -> [(start,end), ...]"""
    if not ts: return []
    out=[]
    s=ts[0]; p=ts[0]
    for t in ts[1:]:
        if t==p+1: p=t
        else:
            out.append((s,p))
            s=t; p=t
    out.append((s,p))
    return out

def load_meta_timestamps(sig):
    path = os.path.join(META_DIR, f"{sig}_meta.json")
    with open(path,"r") as f:
        meta=json.load(f)
    ts = pd.to_datetime([r.get("push_timestamp", None) for r in meta["rows"]], errors="coerce")
    return pd.Series(ts)

def closest_t_index(ts_series, target_ts):
    valid = ts_series.dropna()
    if len(valid)==0: return None
    diffs = (valid - target_ts).abs()
    return int(diffs.idxmin())

# Load
res = pd.read_csv(RESULTS_CSV)
alerts = pd.read_csv(ALERTS_CSV)

res["signature_id"] = res["signature_id"].apply(norm_sig)
alerts["signature_id"] = alerts["signature_id"].apply(norm_sig)
alerts["push_timestamp"] = pd.to_datetime(alerts["push_timestamp"], errors="coerce")
alerts = alerts.dropna(subset=["push_timestamp"])

processed_sigs = sorted(res["signature_id"].unique())

# Build per-signature structures
rows=[]
for sig in processed_sigs:
    df = res[res["signature_id"]==sig].copy()
    df["t"] = df["t"].astype(int)
    T = df["t"].max() + 1

    det_ts = sorted(df.loc[df["decision"]==1, "t"].tolist())
    intervals = extract_intervals(det_ts)

    # counts
    samples_recorded = len(det_ts)                       # paper: Samples Recorded
    unique_changes  = len(intervals)                     # paper: Useful Unique Changes Detected
    enabled_frac    = samples_recorded / T if T>0 else 0
    storage_reduc   = 1 - enabled_frac                   # paper: Storage Overhead Reduced By ARIMA

    # map alerts -> t indices
    ts_series = load_meta_timestamps(sig)
    sig_alerts = alerts[alerts["signature_id"]==sig]
    alert_t = []
    for _, a in sig_alerts.iterrows():
        ta = closest_t_index(ts_series, a["push_timestamp"])
        if ta is not None:
            alert_t.append(ta)

    # paper: Was the bug detected? (here: alert detected)
    detected = False
    delay_list=[]
    for ta in alert_t:
        # hit if any decision interval overlaps ta±TOL
        hit=False
        for (s,e) in intervals:
            if (s - TOL) <= ta <= (e + TOL):
                hit=True
                delay_list.append(max(0, s - ta))  # pushes after alert (0 if before/overlap)
                break
        detected = detected or hit

    avg_delay = float(np.mean(delay_list)) if delay_list else None

    rows.append({
        "signature_id": sig,
        "Was Alert Detected?": "Yes" if detected else "No",
        "Useful Unique Changes Detected": unique_changes,
        "Samples Recorded (decision=1 points)": samples_recorded,
        "Enabled Fraction": enabled_frac,
        "Storage Overhead Reduced (vs always-on)": storage_reduc,
        "Avg Detection Delay (pushes)": avg_delay,
        "Num Alerts": len(alert_t)
    })

out = pd.DataFrame(rows)

print("=== Paper-style per-signature evaluation ===")
display(out)

print("\n=== Aggregate (paper-style) ===")
print("Alerted signatures:", int((out["Num Alerts"]>0).sum()))
print("Detected alerts (signatures):", int(((out["Num Alerts"]>0) & (out["Was Alert Detected?"]=="Yes")).sum()))
print("Mean storage reduction:", float(out["Storage Overhead Reduced (vs always-on)"].mean()))
print("Mean enabled fraction:", float(out["Enabled Fraction"].mean()))
print("Mean unique changes:", float(out["Useful Unique Changes Detected"].mean()))

=== Paper-style per-signature evaluation ===


,signature_id,Was Alert Detected?,Useful Unique Changes Detected,Samples Recorded (decision=1 points),Enabled Fraction,Storage Overhead Reduced (vs always-on),Avg Detection Delay (pushes),Num Alerts
0,4653372,Yes,47,82,0.143608,0.856392,0.00,1
1,4653777,Yes,37,63,0.112100,0.887900,0.00,1
2,4653804,Yes,36,68,0.117444,0.882556,0.00,1
3,4653842,Yes,43,58,0.102293,0.897707,0.00,3
4,4653876,Yes,44,80,0.134228,0.865772,0.00,3
5,4653891,Yes,50,75,0.118110,0.881890,0.00,3
6,4653997,Yes,33,60,0.103093,0.896907,0.00,6
7,4654029,Yes,35,60,0.100334,0.899666,0.25,4
8,4677930,Yes,15,31,0.098101,0.901899,0.00,1
9,4754836,Yes,21,31,0.095092,0.904908,5.00,1



=== Aggregate (paper-style) ===
Alerted signatures: 10
Detected alerts (signatures): 10
Mean storage reduction: 0.8875597066225623
Mean enabled fraction: 0.11244029337743769
Mean unique changes: 36.1


In [ ]:
import pandas as pd
import numpy as np
import os, json

RESULTS_CSV = "mozilla_results_LAST.csv"
ALERTS_CSV  = "alerts_data.csv"
META_DIR    = "vectors"
TOL = 5

def norm_sig(x):
    try: return str(int(float(x)))
    except: return str(x).strip()

def extract_intervals(ts):
    """ts = sorted list of t where decision==1 -> [(start,end), ...]"""
    if not ts: return []
    out=[]
    s=ts[0]; p=ts[0]
    for t in ts[1:]:
        if t==p+1: p=t
        else:
            out.append((s,p))
            s=t; p=t
    out.append((s,p))
    return out

def load_meta_timestamps(sig):
    path = os.path.join(META_DIR, f"{sig}_meta.json")
    with open(path,"r") as f:
        meta=json.load(f)
    ts = pd.to_datetime([r.get("push_timestamp", None) for r in meta["rows"]], errors="coerce")
    return pd.Series(ts)

def closest_t_index(ts_series, target_ts):
    valid = ts_series.dropna()
    if len(valid)==0: return None
    diffs = (valid - target_ts).abs()
    return int(diffs.idxmin())

# Load
res = pd.read_csv(RESULTS_CSV)
alerts = pd.read_csv(ALERTS_CSV)

res["signature_id"] = res["signature_id"].apply(norm_sig)
alerts["signature_id"] = alerts["signature_id"].apply(norm_sig)
alerts["push_timestamp"] = pd.to_datetime(alerts["push_timestamp"], errors="coerce")
alerts = alerts.dropna(subset=["push_timestamp"])

processed_sigs = sorted(res["signature_id"].unique())

# Build per-signature structures
rows=[]
for sig in processed_sigs:
    df = res[res["signature_id"]==sig].copy()
    df["t"] = df["t"].astype(int)
    T = df["t"].max() + 1

    det_ts = sorted(df.loc[df["decision"]==1, "t"].tolist())
    intervals = extract_intervals(det_ts)

    # counts
    samples_recorded = len(det_ts)                       # paper: Samples Recorded
    unique_changes  = len(intervals)                     # paper: Useful Unique Changes Detected
    enabled_frac    = samples_recorded / T if T>0 else 0
    storage_reduc   = 1 - enabled_frac                   # paper: Storage Overhead Reduced By ARIMA

    # map alerts -> t indices
    ts_series = load_meta_timestamps(sig)
    sig_alerts = alerts[alerts["signature_id"]==sig]
    alert_t = []
    for _, a in sig_alerts.iterrows():
        ta = closest_t_index(ts_series, a["push_timestamp"])
        if ta is not None:
            alert_t.append(ta)

    # paper: Was the bug detected? (here: alert detected)
    detected = False
    delay_list=[]
    for ta in alert_t:
        # hit if any decision interval overlaps ta±TOL
        hit=False
        for (s,e) in intervals:
            if (s - TOL) <= ta <= (e + TOL):
                hit=True
                delay_list.append(max(0, s - ta))  # pushes after alert (0 if before/overlap)
                break
        detected = detected or hit

    avg_delay = float(np.mean(delay_list)) if delay_list else None

    rows.append({
        "signature_id": sig,
        "Was Alert Detected?": "Yes" if detected else "No",
        "Useful Unique Changes Detected": unique_changes,
        "Samples Recorded (decision=1 points)": samples_recorded,
        "Enabled Fraction": enabled_frac,
        "Storage Overhead Reduced (vs always-on)": storage_reduc,
        "Avg Detection Delay (pushes)": avg_delay,
        "Num Alerts": len(alert_t)
    })

out = pd.DataFrame(rows)

print("=== Paper-style per-signature evaluation ===")
display(out)

print("\n=== Aggregate (paper-style) ===")
print("Alerted signatures:", int((out["Num Alerts"]>0).sum()))
print("Detected alerts (signatures):", int(((out["Num Alerts"]>0) & (out["Was Alert Detected?"]=="Yes")).sum()))
print("Mean storage reduction:", float(out["Storage Overhead Reduced (vs always-on)"].mean()))
print("Mean enabled fraction:", float(out["Enabled Fraction"].mean()))
print("Mean unique changes:", float(out["Useful Unique Changes Detected"].mean()))

=== Paper-style per-signature evaluation ===


,signature_id,Was Alert Detected?,Useful Unique Changes Detected,Samples Recorded (decision=1 points),Enabled Fraction,Storage Overhead Reduced (vs always-on),Avg Detection Delay (pushes),Num Alerts
0,4653372,Yes,42,89,0.155867,0.844133,0.0,1
1,4653777,Yes,37,86,0.153025,0.846975,0.0,1
2,4653804,No,41,75,0.129534,0.870466,NaN,1
3,4653842,Yes,40,73,0.128748,0.871252,0.0,3
4,4653876,Yes,40,76,0.127517,0.872483,0.0,3
5,4653891,Yes,61,124,0.195276,0.804724,0.0,3
6,4653997,Yes,37,70,0.120275,0.879725,0.0,6
7,4654029,Yes,37,86,0.143813,0.856187,0.0,4
8,4677930,Yes,13,40,0.126582,0.873418,0.0,1
9,4754836,Yes,17,38,0.116564,0.883436,0.0,1



=== Aggregate (paper-style) ===
Alerted signatures: 10
Detected alerts (signatures): 9
Mean storage reduction: 0.8602800026768855
Mean enabled fraction: 0.13971999732311438
Mean unique changes: 36.5


In [ ]:
import pandas as pd
import numpy as np
import os, json

RESULTS_CSV = "mozilla_results_SMA.csv"
ALERTS_CSV  = "alerts_data.csv"
META_DIR    = "vectors"
TOL = 5

def norm_sig(x):
    try: return str(int(float(x)))
    except: return str(x).strip()

def extract_intervals(ts):
    """ts = sorted list of t where decision==1 -> [(start,end), ...]"""
    if not ts: return []
    out=[]
    s=ts[0]; p=ts[0]
    for t in ts[1:]:
        if t==p+1: p=t
        else:
            out.append((s,p))
            s=t; p=t
    out.append((s,p))
    return out

def load_meta_timestamps(sig):
    path = os.path.join(META_DIR, f"{sig}_meta.json")
    with open(path,"r") as f:
        meta=json.load(f)
    ts = pd.to_datetime([r.get("push_timestamp", None) for r in meta["rows"]], errors="coerce")
    return pd.Series(ts)

def closest_t_index(ts_series, target_ts):
    valid = ts_series.dropna()
    if len(valid)==0: return None
    diffs = (valid - target_ts).abs()
    return int(diffs.idxmin())

# Load
res = pd.read_csv(RESULTS_CSV)
alerts = pd.read_csv(ALERTS_CSV)

res["signature_id"] = res["signature_id"].apply(norm_sig)
alerts["signature_id"] = alerts["signature_id"].apply(norm_sig)
alerts["push_timestamp"] = pd.to_datetime(alerts["push_timestamp"], errors="coerce")
alerts = alerts.dropna(subset=["push_timestamp"])

processed_sigs = sorted(res["signature_id"].unique())

# Build per-signature structures
rows=[]
for sig in processed_sigs:
    df = res[res["signature_id"]==sig].copy()
    df["t"] = df["t"].astype(int)
    T = df["t"].max() + 1

    det_ts = sorted(df.loc[df["decision"]==1, "t"].tolist())
    intervals = extract_intervals(det_ts)

    # counts
    samples_recorded = len(det_ts)                       # paper: Samples Recorded
    unique_changes  = len(intervals)                     # paper: Useful Unique Changes Detected
    enabled_frac    = samples_recorded / T if T>0 else 0
    storage_reduc   = 1 - enabled_frac                   # paper: Storage Overhead Reduced By ARIMA

    # map alerts -> t indices
    ts_series = load_meta_timestamps(sig)
    sig_alerts = alerts[alerts["signature_id"]==sig]
    alert_t = []
    for _, a in sig_alerts.iterrows():
        ta = closest_t_index(ts_series, a["push_timestamp"])
        if ta is not None:
            alert_t.append(ta)

    # paper: Was the bug detected? (here: alert detected)
    detected = False
    delay_list=[]
    for ta in alert_t:
        # hit if any decision interval overlaps ta±TOL
        hit=False
        for (s,e) in intervals:
            if (s - TOL) <= ta <= (e + TOL):
                hit=True
                delay_list.append(max(0, s - ta))  # pushes after alert (0 if before/overlap)
                break
        detected = detected or hit

    avg_delay = float(np.mean(delay_list)) if delay_list else None

    rows.append({
        "signature_id": sig,
        "Was Alert Detected?": "Yes" if detected else "No",
        "Useful Unique Changes Detected": unique_changes,
        "Samples Recorded (decision=1 points)": samples_recorded,
        "Enabled Fraction": enabled_frac,
        "Storage Overhead Reduced (vs always-on)": storage_reduc,
        "Avg Detection Delay (pushes)": avg_delay,
        "Num Alerts": len(alert_t)
    })

out = pd.DataFrame(rows)

print("=== Paper-style per-signature evaluation ===")
display(out)

print("\n=== Aggregate (paper-style) ===")
print("Alerted signatures:", int((out["Num Alerts"]>0).sum()))
print("Detected alerts (signatures):", int(((out["Num Alerts"]>0) & (out["Was Alert Detected?"]=="Yes")).sum()))
print("Mean storage reduction:", float(out["Storage Overhead Reduced (vs always-on)"].mean()))
print("Mean enabled fraction:", float(out["Enabled Fraction"].mean()))
print("Mean unique changes:", float(out["Useful Unique Changes Detected"].mean()))

=== Paper-style per-signature evaluation ===


,signature_id,Was Alert Detected?,Useful Unique Changes Detected,Samples Recorded (decision=1 points),Enabled Fraction,Storage Overhead Reduced (vs always-on),Avg Detection Delay (pushes),Num Alerts
0,4653372,Yes,53,77,0.134851,0.865149,0.000000,1
1,4653777,Yes,38,64,0.113879,0.886121,0.000000,1
2,4653804,Yes,40,79,0.136442,0.863558,0.000000,1
3,4653842,Yes,43,67,0.118166,0.881834,0.000000,3
4,4653876,Yes,42,81,0.135906,0.864094,0.000000,3
5,4653891,Yes,54,95,0.149606,0.850394,0.333333,3
6,4653997,Yes,35,63,0.108247,0.891753,0.500000,6
7,4654029,Yes,35,66,0.110368,0.889632,0.000000,4
8,4677930,Yes,15,33,0.104430,0.895570,0.000000,1
9,4754836,Yes,12,46,0.141104,0.858896,5.000000,1



=== Aggregate (paper-style) ===
Alerted signatures: 10
Detected alerts (signatures): 10
Mean storage reduction: 0.8746999602268156
Mean enabled fraction: 0.12530003977318435
Mean unique changes: 36.7


In [ ]:
import shutil
from google.colab import files

# Create zip archive
shutil.make_archive("plots_SMA", "zip", "plots_SMA")

# Download it
files.download("plots_.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>